# Install and Smoke

Confirms importability, top-level surface size, submodule availability, and the first tiny-model capture.

Every `COVERAGE_CALLS` entry below is resolved against the current TorchLens API in this notebook.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
from torch import nn
from _shared import audit_touch_items, summarize_value  # noqa: E402


class AuditBufferModel(nn.Module):
    """Tiny model with a registered buffer for BufferLog coverage."""

    def __init__(self) -> None:
        """Initialize one linear layer and one buffer."""

        super().__init__()
        self.linear = nn.Linear(4, 2)
        self.register_buffer("offset", torch.ones(1, 4))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the buffer before the linear layer."""

        return self.linear(x + self.offset)


model = tiny_model(seed=0).train()
x = torch.randn(1, 4, requires_grad=True)
log = tl.trace(
    model,
    x,
    layers_to_save="all",
    save_grads=True,
    save_arg_values=True,
    save_rng_states=True,
    train_mode=True,
    vis_opt="none",
)
loss = log.layer_list[-1].out.sum()
log.log_backward(loss)
layer_pass = log.layer_list[1]
layer_log = log.layer_logs[layer_pass.layer_label_no_pass]
module_log = next(iter(log.modules))
module_pass = next(iter(module_log.ops.values()))
param_log = next(iter(log.params))
grad_fn_log = next(iter(log.grad_fns))
grad_fn_pass = next(iter(grad_fn_log.ops.values()))

buffer_log_model = AuditBufferModel().eval()
buffer_log_capture = tl.trace(
    buffer_log_model, torch.randn(1, 4), layers_to_save="all", vis_opt="none"
)
buffer_log = next(iter(buffer_log_capture.buffers))

clean, corrupt = make_clean_corrupt_pair(seed=0)
bundle = tl.bundle(
    [
        ("baseline", log),
        ("corrupt", tl.trace(tiny_model(seed=1).eval(), corrupt, vis_opt="none")),
    ],
    baseline="baseline",
)
AUDIT_CONTEXT = {
    "tl": tl,
    "Trace": log,
    "LayerLog": layer_log,
    "OpLog": layer_pass,
    "ModuleLog": module_log,
    "ModuleCallLog": module_pass,
    "ParamLog": param_log,
    "BufferLog": buffer_log,
    "GradFnLog": grad_fn_log,
    "GradFnCallLog": grad_fn_pass,
    "Bundle": bundle,
}
print(
    f"audit context: {len(log.layer_list)} layer ops, {len(log.modules)} modules, {len(log.grad_fns)} grad fns"
)

## Install and Smoke: concrete workflow

This cell demonstrates the user-facing workflow for this notebook before the manifest sweep.

In [ ]:
COVERAGE_CALLS = [
    "tl.__all__",
    "tl.__version__",
    "tl.accessors",
    "tl.bridge",
    "tl.callbacks",
    "tl.capture",
    "tl.compat",
    "tl.constants",
    "tl.constants.BUFFER_LOG_FIELD_ORDER",
    "tl.constants.FUNC_CALL_LOCATION_FIELD_ORDER",
    "tl.constants.GRAD_FN_LOG_FIELD_ORDER",
    "tl.constants.GRAD_FN_PASS_LOG_FIELD_ORDER",
    "tl.constants.IGNORED_FUNCS",
    "tl.constants.LAYER_LOG_FIELD_ORDER",
    "tl.constants.LAYER_PASS_LOG_FIELD_ORDER",
    "tl.constants.List",
    "tl.constants.MODEL_LOG_FIELD_ORDER",
    "tl.constants.MODULE_LOG_FIELD_ORDER",
    "tl.constants.MODULE_PASS_LOG_FIELD_ORDER",
    "tl.constants.ORIG_TORCH_FUNCS",
    "tl.constants.OVERRIDABLE_FUNCS",
    "tl.constants.PARAM_LOG_FIELD_ORDER",
    "tl.constants.TENSOR_LOG_FIELD_ORDER",
    "tl.constants.TORCHVISION_FUNCS",
    "tl.constants.functools",
    "tl.constants.get_ignored_functions",
    "tl.constants.get_testing_overrides",
    "tl.constants.torch",
    "tl.constants.torchvision",
    "tl.constants.types",
    "tl.constants.warnings",
    "tl.data_classes",
    "tl.decoration",
    "tl.errors",
    "tl.examples",
    "tl.experimental",
    "tl.export",
    "tl.fastlog",
    "tl.grad",
    "tl.intervene",
    "tl.intervention",
    "tl.io",
    "tl.llm",
    "tl.multi_trace",
    "tl.neuro",
    "tl.notebook",
    "tl.observers",
    "tl.options",
    "tl.paper",
    "tl.partial",
    "tl.postprocess",
    "tl.report",
    "tl.stats",
    "tl.types",
    "tl.user_funcs",
    "tl.utils",
    "tl.validation",
    "tl.viewer",
    "tl.visualization",
    "tl.viz",
]

inline_show("version", tl.__version__)
inline_show("export count", len(tl.__all__))
imported = [name for name in sorted(tl.__all__)[:8] if hasattr(tl, name)]
inline_show("first exports", imported)
inline_show("first labels", log.layer_labels[:5])

## Install and Smoke coverage cluster 1

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.__all__",
    "tl.__version__",
    "tl.accessors",
    "tl.bridge",
    "tl.callbacks",
    "tl.capture",
    "tl.compat",
    "tl.constants",
    "tl.constants.BUFFER_LOG_FIELD_ORDER",
    "tl.constants.FUNC_CALL_LOCATION_FIELD_ORDER",
    "tl.constants.GRAD_FN_LOG_FIELD_ORDER",
    "tl.constants.GRAD_FN_PASS_LOG_FIELD_ORDER",
    "tl.constants.IGNORED_FUNCS",
    "tl.constants.LAYER_LOG_FIELD_ORDER",
    "tl.constants.LAYER_PASS_LOG_FIELD_ORDER",
    "tl.constants.List",
    "tl.constants.MODEL_LOG_FIELD_ORDER",
    "tl.constants.MODULE_LOG_FIELD_ORDER",
    "tl.constants.MODULE_PASS_LOG_FIELD_ORDER",
    "tl.constants.ORIG_TORCH_FUNCS",
    "tl.constants.OVERRIDABLE_FUNCS",
    "tl.constants.PARAM_LOG_FIELD_ORDER",
    "tl.constants.TENSOR_LOG_FIELD_ORDER",
    "tl.constants.TORCHVISION_FUNCS",
    "tl.constants.functools",
    "tl.constants.get_ignored_functions",
    "tl.constants.get_testing_overrides",
    "tl.constants.torch",
    "tl.constants.torchvision",
    "tl.constants.types",
    "tl.constants.warnings",
    "tl.data_classes",
]

audit_touch_items("Install and Smoke coverage cluster 1", ITEMS, AUDIT_CONTEXT)

## Install and Smoke coverage cluster 2

The helper resolves each listed name against live notebook objects and prints compact samples.

In [ ]:
ITEMS = [
    "tl.decoration",
    "tl.errors",
    "tl.examples",
    "tl.experimental",
    "tl.export",
    "tl.fastlog",
    "tl.grad",
    "tl.intervene",
    "tl.intervention",
    "tl.io",
    "tl.llm",
    "tl.multi_trace",
    "tl.neuro",
    "tl.notebook",
    "tl.observers",
    "tl.options",
    "tl.paper",
    "tl.partial",
    "tl.postprocess",
    "tl.report",
    "tl.stats",
    "tl.types",
    "tl.user_funcs",
    "tl.utils",
    "tl.validation",
    "tl.viewer",
    "tl.visualization",
    "tl.viz",
]

audit_touch_items("Install and Smoke coverage cluster 2", ITEMS, AUDIT_CONTEXT)

In [ ]:
COVERAGE_CALLS = [
    "tl.__all__",
    "tl.__version__",
    "tl.accessors",
    "tl.bridge",
    "tl.callbacks",
    "tl.capture",
    "tl.compat",
    "tl.constants",
    "tl.constants.BUFFER_LOG_FIELD_ORDER",
    "tl.constants.FUNC_CALL_LOCATION_FIELD_ORDER",
    "tl.constants.GRAD_FN_LOG_FIELD_ORDER",
    "tl.constants.GRAD_FN_PASS_LOG_FIELD_ORDER",
    "tl.constants.IGNORED_FUNCS",
    "tl.constants.LAYER_LOG_FIELD_ORDER",
    "tl.constants.LAYER_PASS_LOG_FIELD_ORDER",
    "tl.constants.List",
    "tl.constants.MODEL_LOG_FIELD_ORDER",
    "tl.constants.MODULE_LOG_FIELD_ORDER",
    "tl.constants.MODULE_PASS_LOG_FIELD_ORDER",
    "tl.constants.ORIG_TORCH_FUNCS",
    "tl.constants.OVERRIDABLE_FUNCS",
    "tl.constants.PARAM_LOG_FIELD_ORDER",
    "tl.constants.TENSOR_LOG_FIELD_ORDER",
    "tl.constants.TORCHVISION_FUNCS",
    "tl.constants.functools",
    "tl.constants.get_ignored_functions",
    "tl.constants.get_testing_overrides",
    "tl.constants.torch",
    "tl.constants.torchvision",
    "tl.constants.types",
    "tl.constants.warnings",
    "tl.data_classes",
    "tl.decoration",
    "tl.errors",
    "tl.examples",
    "tl.experimental",
    "tl.export",
    "tl.fastlog",
    "tl.grad",
    "tl.intervene",
    "tl.intervention",
    "tl.io",
    "tl.llm",
    "tl.multi_trace",
    "tl.neuro",
    "tl.notebook",
    "tl.observers",
    "tl.options",
    "tl.paper",
    "tl.partial",
    "tl.postprocess",
    "tl.report",
    "tl.stats",
    "tl.types",
    "tl.user_funcs",
    "tl.utils",
    "tl.validation",
    "tl.viewer",
    "tl.visualization",
    "tl.viz",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")